# 06 · Climate_TemperaturaMinima · auditoría geográfica

Resuelve las estaciones de **Temperatura mínima** contra catálogo IDEAM, DIVIPOLA y polígonos municipales.

In [ ]:
from pathlib import Path
import subprocess
import sys

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/cybercolombia/suelosabio.git'
REPO_REF = 'feature/SCRUM-16'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        remote_ref = f'refs/remotes/origin/{REPO_REF}'
        subprocess.run(['git', 'fetch', '--depth', '1', 'origin', f'+refs/heads/{REPO_REF}:{remote_ref}'], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'checkout', '-B', REPO_REF, remote_ref], cwd=REPO_DIR, check=True)
    PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
else:
    candidatos = [REPO_DIR / 'notebooks' / 'ClimatePipeline', REPO_DIR / 'ClimatePipeline', REPO_DIR]
    PIPELINE_DIR = next((p for p in candidatos if (p / 'DatasetConfig.py').exists()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('No se encontró notebooks/ClimatePipeline.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))


In [ ]:
import json
import time
import pandas as pd
from ClimateGeography import (
    GEOGRAPHY_VERSION,
    auditar_geografia,
    cargar_poligonos_municipales,
    construir_catalogo_ideam_de_referencia,
)
from ClimateProcessingUtils import ahora_proyecto, detectar_commit, escribir_json_atomico, escribir_parquet_atomico, slugificar
from DatasetConfig import cargar_configuracion_datasets

VARIABLE_NOMBRE = 'temperatura_minima'
DATASET_ID = 'afdg-3zpb'
CONSOLIDACION_NOMBRE = 'cierre_temperatura_minima_2024_2025_v1'
EJECUCION_GEOGRAFICA = 'estaciones_temperatura_minima_2024_2025_v3'
CAPA_MUNICIPIOS = 'Boyaca_Cundinamarca_Municipios'
EJECUTAR_AUDITORIA_GEOGRAFICA = False
GUARDAR_RESULTADOS = True
SOBRESCRIBIR_RESULTADOS = False

CONFIG = cargar_configuracion_datasets(in_colab=IN_COLAB)
CLIMATE_INPUT_DIR = CONFIG.processed_root / 'clima_diario_curado' / f'variable={VARIABLE_NOMBRE}' / f'fuente={DATASET_ID}' / f'consolidacion={slugificar(CONSOLIDACION_NOMBRE)}'
OUTPUT_DIR = CONFIG.canonical_geography_root_for(VARIABLE_NOMBRE)
ESTACIONES_PATH = CONFIG.shared_root / 'Estaciones_IDEAM_20260527.csv'
DIVIPOLA_PATH = CONFIG.shared_root / 'Divipola.csv'
GPKG_PATH = CONFIG.geography_source_root / 'Boyaca_Cundinamarca_Municipios.gpkg'
print({'variable':VARIABLE_NOMBRE,'entrada':str(CLIMATE_INPUT_DIR),'salida':str(OUTPUT_DIR),'ejecutar':EJECUTAR_AUDITORIA_GEOGRAFICA})


In [ ]:
if not EJECUTAR_AUDITORIA_GEOGRAFICA:
    print('Auditoría geográfica desactivada. Active la bandera después de revisar las rutas.')
else:
    inicio_reloj=time.perf_counter(); inicio=ahora_proyecto()
    manifest_path=CLIMATE_INPUT_DIR/'manifest.json'
    if not manifest_path.exists(): raise FileNotFoundError(manifest_path)
    manifest_entrada=json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest_entrada.get('estado')!='COMPLETA': raise RuntimeError('La consolidación estación-día no está completa.')
    archivos=sorted(CLIMATE_INPUT_DIR.glob('departamento=*/anio=*/mes=*/observaciones_estacion_dia.parquet'))
    if len(archivos)!=48: raise RuntimeError(f'Se esperaban 48 particiones y existen {len(archivos)}.')
    diario=pd.concat([pd.read_parquet(p) for p in archivos],ignore_index=True)
    referencia_geo = CONFIG.canonical_geography_root
    if ESTACIONES_PATH.exists():
        estaciones = pd.read_csv(ESTACIONES_PATH, dtype='string')
    else:
        estaciones_referencia = pd.read_parquet(
            referencia_geo / 'estaciones_municipio_candidato.parquet'
        )
        estaciones = construir_catalogo_ideam_de_referencia(
            diario, estaciones_referencia
        )
        print('Catálogo IDEAM reconstruido desde geografía de referencia y clima.')
    if DIVIPOLA_PATH.exists():
        divipola = pd.read_csv(DIVIPOLA_PATH, dtype='string')
    else:
        divipola = pd.read_parquet(
            referencia_geo / 'divipola_municipios.parquet'
        )
        print('DIVIPOLA reutilizada desde la geografía canónica de referencia.')
    poligonos=cargar_poligonos_municipales(GPKG_PATH,CAPA_MUNICIPIOS)
    resultado=auditar_geografia(diario,estaciones,divipola,poligonos_municipales=poligonos)
    if GUARDAR_RESULTADOS:
        canonicas=resultado.estaciones_candidatas.loc[resultado.estaciones_candidatas['asignacion_canonica'].astype('boolean').fillna(False)].copy()
        tablas={'catalogo_estaciones_climaticas.parquet':resultado.catalogo_climatico,'estaciones_municipio_candidato.parquet':resultado.estaciones_candidatas,'estaciones_revision.parquet':resultado.estaciones_revision,'estaciones_excluidas.parquet':resultado.estaciones_excluidas,'estaciones_municipio.parquet':canonicas,'divipola_municipios.parquet':resultado.divipola_objetivo,'resumen_geografico.parquet':resultado.resumen}
        for nombre,tabla in tablas.items(): escribir_parquet_atomico(tabla,OUTPUT_DIR/nombre,sobrescribir=SOBRESCRIBIR_RESULTADOS)
        fin=ahora_proyecto()
        escribir_json_atomico({'estado':resultado.metricas['estado'],'geography_version':GEOGRAPHY_VERSION,'variable':VARIABLE_NOMBRE,'commit':detectar_commit(REPO_DIR),'inicio':inicio.isoformat(),'fin':fin.isoformat(),'duracion_segundos':round(time.perf_counter()-inicio_reloj,2),'metricas':resultado.metricas,'entrada':str(CLIMATE_INPUT_DIR)},OUTPUT_DIR/'manifest.json',sobrescribir=True)
        print(f'Geografía guardada en {OUTPUT_DIR}')
    display(resultado.resumen)
